# 01 · EDA: Open Food Facts (D1)
Plan v2 Section 6. Uses the same functions as `training/steps/step2b_eda.py` (`runtime/src/nutriscan/eda.py`), so the notebook and the report figures always agree.

Run steps 1-2 first (`data/processed/loaded.parquet`). Without real data it falls back to the synthetic sample: **numbers from the sample are not results.**

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "runtime" / "src"))
import pandas as pd
from IPython.display import Image, display
from nutriscan import config, eda
from nutriscan.data import clean_products
pd.set_option("display.max_columns", 40)
print("outputs read from / written to:", config.OUT_ROOT)

In [ ]:
src = config.PROCESSED / "loaded.parquet"
raw = pd.read_parquet(src) if src.exists() else pd.read_csv(config.SAMPLE_CSV, dtype={"code": str})
print("source:", src if src.exists() else config.SAMPLE_CSV, "| synthetic:", not config.OFF_META.exists())
display(eda.overview(raw, "OFF raw"))
display(eda.dtypes_table(raw))

## 6.1 Missing values (E1)
Which fields can we rely on, and what must be imputed or dropped?

In [ ]:
display(eda.missing_table(raw).head(15))
display(Image(eda.plot_missing(raw, "eda_E1_off_missing", "E1  OFF: missing values per column")))

## Cleaning (Section 5 rules) and engineered features

In [ ]:
df = eda.add_text_features(clean_products(raw))
print(f"{len(raw):,} -> {len(df):,} rows after dropping empty ingredient lists and duplicates")
display(eda.numeric_summary(df, ["n_ingredients", "n_e_numbers", "n_markers", *config.NUTRIENT_COLS]))

## 6.3 Class balance (E2)
NOVA 4 is expected to dominate; this is why macro-F1 is the headline metric.

In [ ]:
nova = df["nova_group"].dropna().astype(int)
display(eda.class_balance(nova))
print("imbalance ratio:", round(eda.imbalance_ratio(nova), 1))
display(Image(eda.plot_class_distribution(nova, "eda_E2_off_nova", "E2  OFF: NOVA group distribution", colors=eda.NOVA_COLORS)))
tier = df.loc[df["health_tier"].isin(eda.TIER_ORDER), "health_tier"]
display(Image(eda.plot_class_distribution(tier, "eda_E2_off_tier", "E2  OFF: health tier", order=eda.TIER_ORDER, colors=eda.TIER_COLORS)))

## Counts by NOVA group (E3) and ingredient order (E9)

In [ ]:
lab = df[df["nova_group"].notna()].copy()
lab["nova_group"] = lab["nova_group"].astype(int)
display(eda.group_summary(lab, "nova_group", ["n_ingredients", "n_e_numbers", "n_markers", "sugars_100g"]))
display(Image(eda.plot_box_by(lab, ["n_ingredients", "n_e_numbers", "n_markers"], "nova_group", "eda_E3_off_counts_by_nova", "E3  Counts by NOVA group")))
display(Image(eda.plot_box_by(lab, ["sugar_pos", "fat_pos"], "nova_group", "eda_E9_off_position_by_nova", "E9  First sugar / fat position")))

## Additives and vocabulary (E4, E5)

In [ ]:
ta4 = eda.top_additives(lab.loc[lab["nova_group"] == 4, "ingredients_norm"])
display(ta4)
display(Image(eda.plot_top_table(ta4, "additive", "eda_E4_off_top_additives_nova4", "E4  Top additives in NOVA 4")))
display(Image(eda.plot_top_ingredients_by_class(lab, "nova_group", "eda_E5_off_top_ingredients_by_nova", "E5  Top ingredients per NOVA group")))

## Food groups (E6), nutrients (E7, E8), language and country (E11)

In [ ]:
display(Image(eda.plot_share_by_category(lab.dropna(subset=["category"]), "category", "nova_group",
                                        "eda_E6_off_nova_share_by_category", "E6  NOVA share per food group", colors=eda.NOVA_COLORS)))
nuts = [c for c in config.NUTRIENT_COLS if c in df and df[c].notna().any()]
if nuts:
    display(Image(eda.plot_nutrients(df, nuts, "eda_E7_off_nutrient_hist", "E7  Nutrients per 100 g")))
    display(Image(eda.plot_corr(df, nuts + ["n_ingredients", "n_e_numbers"], "eda_E7_off_corr", "E7  Spearman correlation")))
    display(Image(eda.plot_box_by(df[df["health_tier"].isin(eda.TIER_ORDER)], nuts[:4], "health_tier",
                                  "eda_E8_off_nutrients_by_tier", "E8  Nutrients by tier", order=eda.TIER_ORDER)))
for col in ("lang", "countries"):
    if col in df:
        display(Image(eda.plot_counts(df[col].astype(str).str.split(",").str[0], f"eda_E11_off_{col}", f"E11  OFF by {col}")))

## Findings
Write one or two sentences under each plot for the report. `reports/eda_findings.md` (from step 2b) has drafts with the numbers already filled in.